# 条件数・数値健全性 — 適用前・原理・適用・効果

係数のオーダーが揃っていないモデルは、丸め誤差が増幅されてソルバーが迷走しやすい
(数値的に「悪条件」)。係数の max/min 比はレンジの広さを見るだけで悪条件性の正確な
指標ではない。真の条件数 κ(A) は係数行列のSVDの `σ_max/σ_min` で得る。

この notebook は次の型でこの現象と対処を追う。

1. **課題(before)** — 単位選択が悪いモデルで κ(A) を測り、悪条件を確認する
2. **原理(principle)** — 係数スケールの広がりと特異値の分布を図で見る
3. **適用(how)** — 変数の単位を変える(スケーリング)ことで κ(A) を下げる
4. **効果(before/after)** — リスケール前後で `matrix_condition` / `scip_basis_condition` /
   `mk.compare_variants` の求解安定性を比較する

題材は3つ。(1) 単位選択で κ(A) が悪化する例を **notebook内で新規に組む**(原材料 mg と
高級添加剤の混合計画。CLAUDE.mdの方針により、既存サンプルで条件が揃わない部分は
検証可能な最小モデルを自作する)。(2) 実測の裏付けとして
**固定費付き生産計画**(`samples/others/fixed_charge.py`、`03_bigm_indicator.ipynb` と同じ題材)の
loose/tight Big-M で κ(A) がどう変わるかも合わせて確認する
(`experiments/run_condition.py` の実測)。(3) SCIP LP基底 κ が実際に大きくなる実例として
**ユニットコミットメント**(`samples/scheduling/unit_commitment.py`)を使う(小さいtoyモデルでは
基底κが測定不能/自明値になり、静的κ(A)との違いが埋もれてしまうため)。

In [1]:
import sys, pathlib
# リポジトリルート(pyproject.toml を持つ階層)を探して import パスに載せる。
# docs/samples/ が存在するため "samples" 有無では docs で止まる。pyproject.toml を目印にする。
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/others", "samples/scheduling"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

def show(fig):  # 静的サイトに埋め込める形でグラフを表示(CDN の plotly.js を読む)
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
from minlpkit.collectors.static_diag import matrix_condition, scip_basis_condition
import fixed_charge as fc
import unit_commitment as uc

# dataviz 参照パレット(minlpkit.live.plots と統一)
C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

repo root: C:\work_space\mip


## 1. 課題(before) — 単位選択が悪いモデルの κ(A)

原材料と高級添加剤を混ぜて栄養基準・エネルギー基準を満たす、最小費用の配合計画を考える。
原材料は **mg単位**で扱っている(ロットが非常に細かく管理されているため)ため、
栄養・エネルギー含有量は「1mgあたり」の極小係数(0.03〜0.05)になる。一方、添加剤は
1個単位(0〜200個)で、含有量係数は 20〜50 と桁が大きく異なる。この「mg vs 個」という
単位の混在が、原材料の係数**列**を他の係数列から桁違いに引き離し、行列を悪条件にする。

In [2]:
from pyscipopt import Model

def blend_model(rescale: bool) -> Model:
    # 原材料(rescale=False: mg単位/True: g単位)+ 添加剤カプセルの配合計画。
    # 数学的には等価(単位変換のみ)。rescale=True は x を 1000分の1(mg->g)にした変数。
    m = Model(); m.hideOutput()
    if not rescale:
        # 原材料 x: mg単位、0 <= x <= 3,000,000 (=3kg)。含有量は「1mgあたり」で極小
        x = m.addVar(lb=0, ub=3_000_000, name="x_mg")
        cost_x, nutrient_x, energy_x = 0.00006, 0.05, 0.03
    else:
        # 原材料 x: g単位、0 <= x <= 3,000。含有量は「1gあたり」= 1mgあたり*1000
        x = m.addVar(lb=0, ub=3_000, name="x_g")
        cost_x, nutrient_x, energy_x = 0.06, 50.0, 30.0
    y = m.addVar(vtype="I", lb=0, ub=200, name="capsules")
    m.addCons(nutrient_x * x + 50 * y >= 8000, name="nutrient")
    m.addCons(energy_x * x + 20 * y >= 4000, name="energy")
    m.setObjective(cost_x * x + 1500 * y, "minimize")
    m.data = dict(x=x, y=y, rescale=rescale)
    return m

kappa_raw = matrix_condition(blend_model(False))
kappa_res = matrix_condition(blend_model(True))
print(f"raw   (mg単位): kappa(A) = {kappa_raw['kappa']:.3e}  shape={kappa_raw['shape']}")
print(f"rescale(g単位): kappa(A) = {kappa_res['kappa']:.3e}  shape={kappa_res['shape']}")

raw   (mg単位): kappa(A) = 5.800e+03  shape=(2, 2)
rescale(g単位): kappa(A) = 1.252e+01  shape=(2, 2)


mg単位のままだと κ(A) が大きい(悪条件)。g単位に変えるだけ(単位変換=線形の列スケーリング)
で、原材料の係数列(x)が添加剤の係数列(y)と同程度のオーダーに揃い、κ(A) が改善する。
次でこの原理を係数分布と特異値スペクトルで見る。

## 2. 原理(principle) — 係数スケールの広がりと特異値

条件数 κ(A)=σ_max/σ_min は、係数行列 A を SVD したときの最大特異値と最小特異値の比。
係数のオーダーがバラバラだと特異値のスケールもバラバラになり、比が大きくなりやすい
(ただし max/min 係数比そのものとは一致しない — あくまで行列全体の伸縮率)。

In [3]:
from minlpkit.collectors.static_diag import extract_coefficients

def build_A(rescale):
    m = blend_model(rescale)
    vars_ = m.getVars()
    vidx = {v.name: i for i, v in enumerate(vars_)}
    rows = []
    for c in m.getConss():
        if not c.isLinear():
            continue
        row = np.zeros(len(vars_))
        for vn, coef in m.getValsLinear(c).items():
            row[vidx[vn]] = coef
        rows.append(row)
    return np.array(rows)

A_raw, A_res = build_A(False), build_A(True)
sv_raw = np.linalg.svd(A_raw, compute_uv=False)
sv_res = np.linalg.svd(A_res, compute_uv=False)

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
    subplot_titles=("係数の絶対値レンジ(対数)", "特異値スペクトル σ(対数)"))

coefs_raw = np.abs(A_raw[A_raw != 0])
coefs_res = np.abs(A_res[A_res != 0])
fig.add_trace(go.Box(y=coefs_raw, name="raw (mg単位)", marker_color=C["muted"],
    boxpoints="all", jitter=0.5), row=1, col=1)
fig.add_trace(go.Box(y=coefs_res, name="rescale (g単位)", marker_color=C["s1"],
    boxpoints="all", jitter=0.5), row=1, col=1)

fig.add_trace(go.Bar(x=["sigma_max", "sigma_min"], y=[sv_raw[0], sv_raw[-1]],
    marker_color=C["muted"], name="raw", showlegend=False,
    text=[f"{v:.2e}" for v in [sv_raw[0], sv_raw[-1]]], textposition="outside"), row=1, col=2)
fig.add_trace(go.Bar(x=["sigma_max", "sigma_min"], y=[sv_res[0], sv_res[-1]],
    marker_color=C["s1"], name="rescale", showlegend=False,
    text=[f"{v:.2e}" for v in [sv_res[0], sv_res[-1]]], textposition="outside"), row=1, col=2)

fig.update_yaxes(type="log", gridcolor=C["grid"], linecolor=C["axis"], row=1, col=1)
fig.update_yaxes(type="log", gridcolor=C["grid"], linecolor=C["axis"], row=1, col=2)
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=420,
    margin=dict(l=50, r=20, t=56, b=40),
    title=dict(text="係数レンジと特異値スペクトル: raw(mg単位) vs rescale(g単位)",
               x=0.01, font=dict(color=C["ink"], size=15)))
show(fig)

raw(mg単位、グレー)は原材料の係数(0.03〜0.05)が添加剤の係数(20〜50)から3桁近く
離れており、最小特異値が極端に小さいため κ(A) が悪化する。rescale(g単位、青)は
原材料の係数が1000倍されて添加剤側と同程度のオーダー(20〜50)に揃い、特異値の比も
大きく縮む。**単位を変えるだけで行列の中身(問題そのもの)は変わらないのに、数値的な
扱いやすさが大きく変わる**ことが分かる。

## 3. 適用(how) — リスケール(単位変換)

`matrix_condition`(solve前、SVDベース)と `scip_basis_condition`(solve後、SCIP LP基底の
`getCondition()`)の2つで、定式化そのものの悪条件と実際に解いた基底の不安定度を
別々に測れる。まず既定設定(presolveあり)で両方を測ってみる。

In [4]:
for rescale, label in [(False, "raw (mg単位)"), (True, "rescale (g単位)")]:
    m = blend_model(rescale)
    kappa = matrix_condition(blend_model(rescale))["kappa"]
    basis_k = scip_basis_condition(m)
    print(f"{label:16s}: kappa(A)={kappa:.3e}   SCIP基底kappa={basis_k:.3e}")

raw (mg単位)      : kappa(A)=5.800e+03   SCIP基底kappa=1.000e+99
rescale (g単位)   : kappa(A)=1.252e+01   SCIP基底kappa=1.000e+99


**正直な注意点**: この2変数の小さいモデルは presolve だけで解けてしまい、SCIPが
最終LP基底を保持しないため `getCondition()` が「測定不可」を表す `1e+99` を返す。
静的 κ(A) は presolve 前の定式化そのものを見るので測定できるが、SCIP基底κは
「実際に何らかのLPを最後まで解いた」ときにしか意味を持たない。presolveを切って
確認する。

In [5]:
def _set_no_presolve(m):
    m.setParam("presolving/maxrounds", 0)  # presolveでLPが消えないようにする
    return m

for rescale, label in [(False, "raw (mg単位)"), (True, "rescale (g単位)")]:
    basis_k = scip_basis_condition(_set_no_presolve(blend_model(rescale)))
    print(f"{label:16s}: SCIP基底kappa(presolveなし) = {basis_k:.3e}")

raw (mg単位)      : SCIP基底kappa(presolveなし) = 1.000e+00
rescale (g単位)   : SCIP基底kappa(presolveなし) = 1.000e+00


presolveを切ると両方とも基底κ≈1と非常に小さい — この2変数モデルは変数が少なく
最適基底自体は単純(ほぼ対角)なため、**静的κ(A)が悪くても実際に解いた基底は安定**という
ケースがありうる。静的κ(A)と基底κは相補的な指標であり、必ずしも一致しない。
基底κが実際に大きくなる実例として、**ユニットコミットメント**
(`samples/scheduling/unit_commitment.py`、変数・制約数が多く時間結合がある大きめのMILP)を見る。

In [6]:
uc_kappa = matrix_condition(uc.build_model())["kappa"]
uc_basis = scip_basis_condition(uc.build_model())
print(f"unit_commitment: 静的kappa(A)={uc_kappa:.3e}   SCIP基底kappa={uc_basis:.3e}")

unit_commitment: 静的kappa(A)=9.592e+02   SCIP基底kappa=2.635e+11


unit_commitment は基底κ≈10^11 と極端に大きく、実際に数値不安定リスクがある領域
(SCIP 10 の厳密有理MILPモードなどが効く対象)。トイモデルでは埋もれていた
「基底κが実際に問題化するケース」がここで確認できる。

最適値・解自体は単位変換で変わらないことも確認しておく(mg版のxをgに換算すれば一致)。

In [7]:
m_raw = blend_model(False); m_raw.optimize()
m_res = blend_model(True); m_res.optimize()
x_raw, x_res = m_raw.data["x"], m_res.data["x"]
print(f"raw    : obj={m_raw.getObjVal():.4f}  x={m_raw.getVal(x_raw):.1f}mg  "
      f"y={m_raw.getVal(m_raw.data['y']):.0f}")
print(f"rescale: obj={m_res.getObjVal():.4f}  x={m_res.getVal(x_res):.4f}g "
      f"(={m_res.getVal(x_res)*1000:.1f}mg)  y={m_res.getVal(m_res.data['y']):.0f}")

raw    : obj=9.6000  x=160000.0mg  y=0
rescale: obj=9.6000  x=160.0000g (=160000.0mg)  y=0


目的値・添加剤個数・原材料量(換算後)がすべて一致する — リスケールは等価変換であり、
数値的な扱いやすさだけを改善する。

## 4. 効果(before/after) — 求解安定性の比較

`mk.compare_variants` でルート双対境界・gap・ノード数を比較する(小規模モデルなので
差は小さいこともあるが、κ(A) 自体の改善が本質)。加えて `03_bigm_indicator.ipynb` と同じ
`fixed_charge` の loose/tight Big-M、および `unit_commitment` の κ(A) の実測差
(`experiments/run_condition.py` の追試)を並べ、規模が大きいモデルでは条件数の差が
そのまま悪条件リスクに直結することを示す。

In [8]:
df = mk.compare_variants(
    {"raw (mg単位)":   lambda: blend_model(False),
     "rescale (g単位)": lambda: blend_model(True)},
    time_limit=10)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

,variant,root_dual,final_dual,final_gap,nodes
0,raw (mg単位),9.6,9.6,0.0,1
1,rescale (g単位),9.6,9.6,0.0,1


In [9]:
names = ["blend raw\n(mg単位)", "blend rescale\n(g単位)",
         "fixed_charge\nloose Big-M", "fixed_charge\ntight Big-M", "unit_\ncommitment"]
kappas = [matrix_condition(blend_model(False))["kappa"],
          matrix_condition(blend_model(True))["kappa"],
          matrix_condition(fc.build_model("loose"))["kappa"],
          matrix_condition(fc.build_model("tight"))["kappa"],
          uc_kappa]
# SCIP基底kappa: presolveが解いてしまうtoy blendモデルは presolveなしで測る。それ以外は既定設定。
scip_ks = [scip_basis_condition(_set_no_presolve(blend_model(False))),
           scip_basis_condition(_set_no_presolve(blend_model(True))),
           scip_basis_condition(fc.build_model("loose")),
           scip_basis_condition(fc.build_model("tight")),
           uc_basis]

fig = go.Figure(layout=base_layout(
    "条件数 kappa(A) 比較 — 大きいほど悪条件(丸め誤差でソルバーが迷走)",
    "", "条件数(対数)", height=440))
fig.add_trace(go.Bar(x=names, y=kappas, name="静的 kappa(A)(係数行列, solve前)",
    marker_color=C["s1"], text=[f"{k:.1e}" for k in kappas], textposition="outside",
    hovertemplate="%{x}<br>kappa(A)=%{y:.2e}<extra></extra>"))
fig.add_trace(go.Bar(x=names, y=scip_ks, name="SCIP LP基底 kappa(solve後)",
    marker_color=C["s2"], text=[f"{k:.1e}" for k in scip_ks], textposition="outside",
    hovertemplate="%{x}<br>基底kappa=%{y:.2e}<extra></extra>"))
fig.add_hline(y=1e7, line=dict(color=C["warn"], width=2, dash="dash"),
              annotation_text="数値的に要注意 (1e7)", annotation_position="top left",
              annotation_font=dict(color=C["warn"], size=11))
fig.update_yaxes(type="log")
fig.update_layout(barmode="group")
show(fig)

In [10]:
print(f"blend        : raw kappa(A)={kappas[0]:.2e} -> rescale kappa(A)={kappas[1]:.2e} "
      f"({kappas[0]/kappas[1]:.0f}倍改善)")
print(f"fixed_charge : loose kappa(A)={kappas[2]:.2e} -> tight kappa(A)={kappas[3]:.2e} "
      f"({kappas[2]/kappas[3]:.0f}倍改善)")
print(f"unit_commitment: 静的kappa(A)={kappas[4]:.2e}   SCIP基底kappa={scip_ks[4]:.2e}"
      f"(数値的に要注意ラインの1e7を大きく超える)")

blend        : raw kappa(A)=5.80e+03 -> rescale kappa(A)=1.25e+01 (463倍改善)
fixed_charge : loose kappa(A)=3.54e+04 -> tight kappa(A)=3.18e+01 (1111倍改善)
unit_commitment: 静的kappa(A)=9.59e+02   SCIP基底kappa=2.64e+11(数値的に要注意ラインの1e7を大きく超える)


## まとめ

- **単位変換(リスケール)だけで κ(A) が桁違いに改善する。** 数学的には完全に等価な変換
  (最適値も解も一致)であり、コストはほぼゼロ。
- Big-M の tight化(`03_bigm_indicator.ipynb`)も広い意味では「係数スケールを揃える」
  リスケールの一種であり、両者は同じ問題(悪条件)への異なる切り口の対処になっている。
- **静的κ(A)とSCIP基底κは一致しないことがある。** 変数の少ない単純なモデルでは静的κ(A)が
  悪くても最適基底は安定なことがあり(セクション3のtoy例)、逆に unit_commitment のように
  規模と時間結合が絡むと基底κだけが極端に悪化することもある。両方を測って初めて
  「定式化の質」と「実際の求解安定性」を分けて診断できる。

### なぜ SCIP が自動でやらないのか

SCIP はLP緩和を解く際に内部でスケーリングを試みる(`NORMALIZE`等)が、これは
**行・列単位の数値的な正規化**であり、モデラーが意図する「変数の単位」までは踏み込まない。
mg か g かはモデルの意味論(何の量を表す変数か)に依存する選択であり、ビルド済みの
係数行列だけからは「どちらが正しい単位か」をソルバーは判断できない。だから
モデラーが単位選びの段階で意識する必要がある。

### 効かないとき・注意

- 係数の max/min 比だけで「数値問題あり」と早合点しないこと。自然なコスト差(~1e3程度)は
  数値問題ではなく、真の悪条件は presolve後で1e6以上のレンジと見るべき
  ([FINDINGS §1](../../playbook/08-condition.md))。
- 条件数診断そのものは**症状を数値化する道具**であり、それ単体では解決策にならない。
  具体的な打ち手は「単位の見直し」や「Big-M排除」([3. Big-M排除](03_bigm_indicator.ipynb))
  である。

関連: [プレイブック 8. 条件数・数値健全性](../../playbook/08-condition.md) /
[プレイブック 3. Big-M排除](../../playbook/03-bigm.md) /
`experiments/run_condition.py` → [`condition.html`](../../gallery/condition.html)。